Transformers 라이브러리

인코더 기반 모델
1. klue/bert-base 기본 모델: 파인 튜닝 전 모델 분류(감성분석)
2. klue/bert-base 기본 모델: 파인 튜닝 후 모델 분류(감성분석)
- 학습용 데이터셋 준비
- 데이터셋 전처리(토크나이징), 샘플링
- 트레이너(Trainer) 설정 
- 학습 실행
- 모델 저장
- 모델 로드 및 모델 추론
- 허깅페이스 모델 업로드

In [4]:
# 인코더 기반 모델
# AutoTokenizer, AutoModelForSequenceClassification
# 일반적인 사전학습 모델은 미세조정이 진행되지 않은 사전학습 모델 가중치를 불러오게 됨
# 이런 경우 미세조정에 사용되는 설정 값이 모델에 포함되어 있지 않기 때문에 추가 설정 값이 필요할 경우가 있음
# 대표 분류 태스크는 분류할 클래스 개수를 뜻하는 num_labels 값을 입력해야 되며, 입력하지 않을시 기본값은 2가 들어가게 됨
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 모델 이름 지정
model_name = "klue/bert-base"
# 사전 학습 모델 로드
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 모델 확인
# (classifier): Linear(in_features=768, out_features=2, bias=True) out_features=2: 분류할 클래스 개수는 2
print(f"model: {model}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model: BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

In [5]:
# model.config.id2label 에서는 해당 모델이 지닌 클래스를 볼 수 있음
# 모델 클래스 확인: {0: 'LABEL_0', 1: 'LABEL_1'} 2개의 클래스 확인
print(f"model.config.id2label: {model.config.id2label}")

model.config.id2label: {0: 'LABEL_0', 1: 'LABEL_1'}


In [6]:
# 인코더 기반 모델
# 1. klue/bert-base 기본 모델: 파인 튜닝 전 모델 분류(감성분석)
# klue/bert-base 순정 모델의 한계: 감성 분석을 하도록 학습된 모델이 아니라, 한국어 문장의 맥락을 이해하도록 사전 학습된 기본 모델
# 무작위 분류기(Classifier) 장착: num_labes=2를 주고 모델을 불러오면, 
# 허깅페이스는 2개 클래스로 분류하고 싶구나 하고 모델 맨 끝에 새로운 분류기 레이어를 무작위(Random) 초기화해서 억지로 붙여준다
# 모델 입장에서는 0번이 무슨 뜻이고 1번이 무슨 뜻인지 전혀 학습된 적이 없기 때문에, 모델 임의로 붙인 LABEL_0/LABEL_1 랜던하게 반환하게 된다
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

# 모델 이름 지정
model_name = "klue/bert-base"
# 2. 사전 학습 모델: 감성 분류
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 3. 모델 -> 디바이스 이동
model.to(device)

# 4. 평가/추론시 설정: 드롭아웃 비활성화, 정규화 레이어 고정
model.eval()

# 5. 입력 텍스트 준비
inputs_text = "안녕? 내 강아지는 귀여요."

# 6. 토크나이저
inputs = tokenizer(
    inputs_text, # 텍스트가 텍스트를 쪼개고, 숫자로 변환
    return_tensors="pt", # 텐서 형태 변환
    truncation=True, # 글자수 초과시 자르기
    padding=True, # 가장 긴 문장을 기준으로 짧은 문장의 빈자리를 빈 토큰(padding token)으로 채운다
    max_length=512, # 최대 허용 길이 제한
# 토크나이징이 끝나서 만들어진 텐서 데이터(input_ids, attention_mask 등)를 모델이 올라가 있는 동일한 하드웨어로 전송
).to(device)

# 7. 미분 계산 비활성화 상태에서 추론 실행: 미분(역전파) 계산을 막아서 메모리를 아끼고 속도를 높임
with torch.no_grad():
    # 모델의 전체 출력값
    outputs = model(**inputs)
    print(f"outputs: {outputs}")

    # 로짓(raw score) 값 추출
    logits = outputs.logits
    print(f"logits: {logits}")

# 로짓에서 가장 확률이 높은 최종 예측 클래스(인데스) 추출
# torch.argmax(logits, dim=-1): 텐서에서 가장 큰값이 위치한 인덱스(몇번째 칸인지) 찾아 변환
# dim=-1: 다차원 텐서에서 어떤 방향(차원)을 기준으로 최대값을 찾을 것인가 지정, 가장 마지막 차원 의미
# .item(): 텐서 형태 -> int/float 기본형 데이터로 변환 예) 순수한 숫자 1로 변환
pred_class_id = torch.argmax(input=logits, dim=-1).item()
print(f"pred_class_id: {pred_class_id}")

# 필요시 확률값으로 변환 할 수 있음, 최종합이 1
# 첫번째 문장에 대한 모든 클래스(2개) 확률 리스트
probabilites = torch.softmax(input=logits, dim=-1)
print(f"probabilites: {probabilites}")

# probabilites[0]: 첫번째 문장의 전체 확률 목록을 가져온다.
# [pred_class_id]: 그 2개 확률 목록중에서, 이전에 torch.softmax로 구했던 가장 점수가 높았던 최종 예측 클래스 번호(인덱스)에 해당하는 확률값만 집어낸다
# .item(): 그 확률값 텐서를 기본형 float 형태로 꺼내온다
pred_prob = probabilites[0][pred_class_id].item()
print(f"pred_prob: {pred_prob}")

model_class=model.config.id2label[pred_class_id]

# 최종 모델 추론 결과 확인
print(f"입력 텍스트: {inputs_text}")
print(f"예측된 클래스 번호(Label ID): {pred_class_id}")
print(f"예측 확신도(Probability): {pred_prob:.4f}")
print(f"예측 클래스(model_class): {model_class}")

device: cpu


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


outputs: SequenceClassifierOutput(loss=None, logits=tensor([[ 0.0854, -0.4471]]), hidden_states=None, attentions=None)
logits: tensor([[ 0.0854, -0.4471]])
pred_class_id: 0
probabilites: tensor([[0.6301, 0.3699]])
pred_prob: 0.6300574541091919
입력 텍스트: 안녕? 내 강아지는 귀여요.
예측된 클래스 번호(Label ID): 0
예측 확신도(Probability): 0.6301
예측 클래스(model_class): LABEL_0


In [ ]:
# 인코더 기반 모델
# 2. klue/bert-base 기본 모델: 파인 튜닝 후 모델 분류(감성분석)
# 학습용 데이터셋 준비
# - 학습용 데이터셋: 표준 감성 분석 데이터셋 NSMC(Naver Sentiment Movie Corpus) 
# - 데이터를 불러와 토크나이징을 수행하는 작업
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset

# 1. 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

# 2. 사전학습 모델 및 토크나이저 로드: num_labels=2 로 감성 분류 설정
model_name = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to(device) # 모델 -> 디바이스(cpu/gpu) 이동

# 3. 네이버 영화 리뷰(NSMC) 감성 분류 데이터셋 로드
# 훈련용(train), 평가용(test) 데이터 일부를 가져온다
dataset = load_dataset(
    path="nsmc",
    split="train[:2500]",
    trust_remote_code=True, # 라이브러리의 최신 보안 정책때문에 외부 데이터셋 스크립트를 실행할때 "이 코드를 신뢰할 수 있는지 확인했다"
) # 실습용으로 2,000개만 샘플링
eval_dataset = load_dataset(
    path="nsmc", 
    split="test[:600]",
    trust_remote_code=True,
) # 평가용으로 500개만 샘플링

print(f"dataset: {dataset}")

device: cpu


/Users/ai/deep-learning/LLM/rag_pdf_qdrant/venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


dataset: Dataset({
    features: ['id', 'document', 'label'],
    num_rows: 2500
})


In [12]:
# 4. 데이터 전처리 함수: 텍스트를 모델이 읽을 수 있는 토큰 ID 변환
def tokenize_function(examples):
    return tokenizer(
        examples["document"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

# 5. 데이터셋 전체에 토크나이징 일괄 적용
encoded_train_dataset = dataset.map( # train
    function=tokenize_function,
    batched=True,
    batch_size=1000,
)
encoded_eval_dataset = dataset.map( # test
    function=tokenize_function,
    batched=True,
    batch_size=1000,
)

print("데이터 전처리 완료! 학습용 샘플 예시:")
print(encoded_train_dataset[0])

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

데이터 전처리 완료! 학습용 샘플 예시:
{'id': '9976970', 'document': '아 더빙.. 진짜 짜증나네요 목소리', 'label': 0, 'input_ids': [2, 1376, 831, 2604, 18, 18, 4229, 9801, 2075, 2203, 2182, 4243, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,

In [13]:
from transformers import Trainer, TrainingArguments

# 6. 학습 하이퍼파라미터 및 설정 정의
training_args = TrainingArguments(
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    num_train_epochs=3,
    eval_strategy="epoch",
    # evaluation_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,
    logging_dir="./content/logs",
    save_strategy="epoch",
    output_dir="./content/bert_base__sentiment_classification_ckpt",
    report_to="tensorboard",
)

# 7. Trainer 객체 초기화
trainer = Trainer(
    model=model, # 감성 분류 모델
    args=training_args, # 학습 하이퍼파라미터 설정
    train_dataset=encoded_train_dataset, # 훈련용
    eval_dataset=encoded_eval_dataset, # 평가용
    tokenizer=tokenizer # 토크나이저
)

In [14]:
# 8. 모델 학습(파인튜닝)
print("모델 파인튜닝(학습)을 시작합니다...")
trainer.train()
print("학습이 완료되었습니다!")

모델 파인튜닝(학습)을 시작합니다...


  0%|          | 0/471 [00:00<?, ?it/s]

{'loss': 0.7149, 'grad_norm': 11.24641227722168, 'learning_rate': 1.9575371549893843e-05, 'epoch': 0.06}
{'loss': 0.6115, 'grad_norm': 5.962913513183594, 'learning_rate': 1.9150743099787688e-05, 'epoch': 0.13}
{'loss': 0.5413, 'grad_norm': 5.858605861663818, 'learning_rate': 1.872611464968153e-05, 'epoch': 0.19}
{'loss': 0.4578, 'grad_norm': 9.355749130249023, 'learning_rate': 1.8301486199575375e-05, 'epoch': 0.25}
{'loss': 0.4003, 'grad_norm': 10.501592636108398, 'learning_rate': 1.7876857749469216e-05, 'epoch': 0.32}
{'loss': 0.3352, 'grad_norm': 7.821881294250488, 'learning_rate': 1.7452229299363058e-05, 'epoch': 0.38}
{'loss': 0.3751, 'grad_norm': 3.0674045085906982, 'learning_rate': 1.7027600849256903e-05, 'epoch': 0.45}
{'loss': 0.4106, 'grad_norm': 10.808830261230469, 'learning_rate': 1.6602972399150744e-05, 'epoch': 0.51}
{'loss': 0.4759, 'grad_norm': 13.650571823120117, 'learning_rate': 1.617834394904459e-05, 'epoch': 0.57}
{'loss': 0.4971, 'grad_norm': 7.475924491882324, 'lea

  0%|          | 0/157 [00:00<?, ?it/s]

{'eval_loss': 0.22594670951366425, 'eval_runtime': 341.8569, 'eval_samples_per_second': 7.313, 'eval_steps_per_second': 0.459, 'epoch': 1.0}
{'loss': 0.4287, 'grad_norm': 5.659934043884277, 'learning_rate': 1.3205944798301487e-05, 'epoch': 1.02}
{'loss': 0.1911, 'grad_norm': 8.342817306518555, 'learning_rate': 1.2781316348195329e-05, 'epoch': 1.08}
{'loss': 0.1763, 'grad_norm': 7.681780815124512, 'learning_rate': 1.2356687898089172e-05, 'epoch': 1.15}
{'loss': 0.2695, 'grad_norm': 8.886374473571777, 'learning_rate': 1.1932059447983015e-05, 'epoch': 1.21}
{'loss': 0.3079, 'grad_norm': 1.4949465990066528, 'learning_rate': 1.1507430997876858e-05, 'epoch': 1.27}
{'loss': 0.2687, 'grad_norm': 16.28136444091797, 'learning_rate': 1.1082802547770702e-05, 'epoch': 1.34}
{'loss': 0.4214, 'grad_norm': 12.33015251159668, 'learning_rate': 1.0658174097664545e-05, 'epoch': 1.4}
{'loss': 0.2695, 'grad_norm': 4.922896385192871, 'learning_rate': 1.0233545647558387e-05, 'epoch': 1.46}
{'loss': 0.4208, 'g

  0%|          | 0/157 [00:00<?, ?it/s]

{'eval_loss': 0.13720488548278809, 'eval_runtime': 341.5731, 'eval_samples_per_second': 7.319, 'eval_steps_per_second': 0.46, 'epoch': 2.0}
{'loss': 0.1896, 'grad_norm': 5.300118923187256, 'learning_rate': 6.411889596602973e-06, 'epoch': 2.04}
{'loss': 0.1792, 'grad_norm': 13.489623069763184, 'learning_rate': 5.987261146496816e-06, 'epoch': 2.1}
{'loss': 0.1552, 'grad_norm': 10.105074882507324, 'learning_rate': 5.562632696390658e-06, 'epoch': 2.17}
{'loss': 0.0491, 'grad_norm': 1.005984902381897, 'learning_rate': 5.1380042462845016e-06, 'epoch': 2.23}
{'loss': 0.0862, 'grad_norm': 16.246980667114258, 'learning_rate': 4.713375796178344e-06, 'epoch': 2.29}
{'loss': 0.1519, 'grad_norm': 2.2615442276000977, 'learning_rate': 4.288747346072187e-06, 'epoch': 2.36}
{'loss': 0.1513, 'grad_norm': 5.136224269866943, 'learning_rate': 3.8641188959660305e-06, 'epoch': 2.42}
{'loss': 0.167, 'grad_norm': 13.98400592803955, 'learning_rate': 3.4394904458598725e-06, 'epoch': 2.48}
{'loss': 0.1331, 'grad_

  0%|          | 0/157 [00:00<?, ?it/s]

{'eval_loss': 0.1019861102104187, 'eval_runtime': 314.1106, 'eval_samples_per_second': 7.959, 'eval_steps_per_second': 0.5, 'epoch': 3.0}
{'train_runtime': 4969.7595, 'train_samples_per_second': 1.509, 'train_steps_per_second': 0.095, 'train_loss': 0.2909329814832428, 'epoch': 3.0}
학습이 완료되었습니다!


In [15]:
# 9. 모델 저장 
save_path = "./content/drive/MyDrive/trained_bert_base_sentiment_model"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f"모델이 안전하게 저장되었습니다: {save_path}")

모델이 안전하게 저장되었습니다: ./content/drive/MyDrive/trained_bert_base_sentiment_model


In [1]:
# # 인코더 기반 모델
# 2. klue/bert-base 기본 모델: 파인 튜닝 후 모델 분류(감성분석)
# 모델 로드 및 모델 추론
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# 2. 모델 이름 지정 및 사전 학습 모델 로드
model_name = "./content/drive/MyDrive/trained_bert_base_sentiment_model"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# 3. 모델을 지정한 디바이스로 이동: BERT 가중치/편향 값을 하드웨어의 메모리에 적재(로드)
model.to(device)

# 4. 평가 모드 전환: 드롭아웃 비활성화/정규화 레이어 고정
model.eval()

# 5. 테스트할 샘플 문장들 입력
inputs_text = [
    "진짜 돈이 하나도 안 아까울 정도로 너무 재미있게 봤습니다!",
    "스토리가 너무 개판이고 시간 아까운 영화네요.",
    "그냥 평범한 영화였습니다. 기대는 안 하는 게 좋아요."
]

# 6. 토크나이징 설정
inputs = tokenizer(
    inputs_text, # 토크나이저가 텍스트를 쪼개고, 숫자로 변환
    return_tensors="pt", # 텐서 형태 변환
    truncation=True, # 글자수 초과시 자르기
    padding=True, # 가장 긴 문장을 기준으로 짧은 문장의 빈자리를 빈 토큰(padding token)으로 채운다
    max_length=128, # 최대 허용 길이 제한
# 토크나이징이 끝나서 만들어진 텐서 데이터(input_ids, attention_mask 등)를 모델이 올라가 있는 동일한 하드웨어로 전송
).to(device)

# 7. 미분 계산 비활성화 상태에서 추론 실행: 미분(역전파) 계산을 막아서 메모리를 아끼고 속도를 높임
with torch.no_grad():
    # 순전파(Forward)
    outputs = model(**inputs) # 모델의 전체 출력값
    print(f"outputs: {outputs}")
    logits = outputs.logits # 로짓(raw score) 값 추출
    print(f"logits: {logits}")

# 8. 소프트맥스(Softmax)를 적용하여 확률값 0~1 사이 값으로 변환
probabilites = torch.softmax(
    input=logits, # 로짓값
    dim=-1, # 마지막 차원값
)
print(f"probabilites: {probabilites}") # 문장 3개

# 9. 결과 해석 및 출력
# NSMC 기준: LABEL_0: 부정, LABEL_1: 긍정
label_map = {0: "부정", 1: "긍정"}

print("\n--- 실무형 순수 추론 결과 ---")
for i, sentence in enumerate(inputs_text):
    # 해당 문장의 로짓값
    probs = probabilites[i]
    print(f"probs: {probs}")

    # 가장 확률이 높은 인덱스 선택
    pred_label_id = torch.argmax(
        input=probs, # 반복문으로 가져온 로짓값
        dim=-1, # 마지막 차원값
    ).item() # float 형 변환
    print(f"pred_label_id: {pred_label_id}")

    # 해당 클래스의 확률(확신도)
    confidence = probs[pred_label_id].item() * 100
    print(f"confidence: {confidence}")

    # NSMC 기준 라벨
    pred_label_name = label_map[pred_label_id]
    print(f"pred_label_name: {pred_label_name}")

    # 보기 좋게 결과 출력
    print(f"문장: '{sentence}'")
    print(f"예측: {pred_label_name} (확신도: {confidence:.2f}%)")
    print(f"[상세 확률] 부정: {probs[0].item() * 100:.2f}% | 긍정: {probs[1].item() * 100:.2f}%\n")

Device: cpu
outputs: SequenceClassifierOutput(loss=None, logits=tensor([[-2.6829,  3.1324],
        [ 2.9190, -2.6278],
        [ 1.7981, -1.7228]]), hidden_states=None, attentions=None)
logits: tensor([[-2.6829,  3.1324],
        [ 2.9190, -2.6278],
        [ 1.7981, -1.7228]])
probabilites: tensor([[0.0030, 0.9970],
        [0.9961, 0.0039],
        [0.9713, 0.0287]])

--- 실무형 순수 추론 결과 ---
probs: tensor([0.0030, 0.9970])
pred_label_id: 1
confidence: 99.70270991325378
pred_label_name: 긍정
문장: '진짜 돈이 하나도 안 아까울 정도로 너무 재미있게 봤습니다!'
예측: 긍정 (확신도: 99.70%)
[상세 확률] 부정: 0.30% | 긍정: 99.70%

probs: tensor([0.9961, 0.0039])
pred_label_id: 0
confidence: 99.61153268814087
pred_label_name: 부정
문장: '스토리가 너무 개판이고 시간 아까운 영화네요.'
예측: 부정 (확신도: 99.61%)
[상세 확률] 부정: 99.61% | 긍정: 0.39%

probs: tensor([0.9713, 0.0287])
pred_label_id: 0
confidence: 97.12784886360168
pred_label_name: 부정
문장: '그냥 평범한 영화였습니다. 기대는 안 하는 게 좋아요.'
예측: 부정 (확신도: 97.13%)
[상세 확률] 부정: 97.13% | 긍정: 2.87%



In [ ]:
# # 인코더 기반 모델
# 3. 허깅페이스 모델 업로드
# 직접 미세조정한 모델을 push_to_hub 메서드를 이용하여 모델 업로드
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. 모질 이름 지정 및 사전학습 모델 로드
model_name = "./content/drive/MyDrive/trained_bert_base_sentiment_model"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# 2. 허깅페이스 리포지터리 이름 지정
USERNAME = "CHY760MARIN"
REPO_NAME = f"{USERNAME}/my-bert-sentiment-fine-tuned"

# 3. 허깅페이스 토큰: 업로드시 토큰 제거
AUTH_TOKEN = ""

# 4. 허깅페이스 허브 등록: model
print("모델 업로드 중...")
model.push_to_hub(
    REPO_NAME, # 리포지터리 이름
    use_temp_dir=True,
    token=AUTH_TOKEN, # 허깅페이스 토큰
)

# 5. 허깅페이스 허브 등록: tokenizer
tokenizer.push_to_hub(
    REPO_NAME, # 리포지터리 이름
    use_temp_dir=True,
    token=AUTH_TOKEN, # 허깅페이스 토큰
)
print("업로드 완료!")

모델 업로드 중...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

업로드 완료!
